# CIDS · 11 — S3e: memory clue for the learned jury (idea from cache-augmented generation)

**Question:** does *"what does this candidate look like compared with KNOWN real tumours and KNOWN false blobs"* add information beyond the 11 clues of the S3d jury (test Dice **0.735**)?

- Same automatic pipeline and candidates as S3d (SegResNet-100 → candidates → SAM-S2b crop + box each).
- Per candidate, a **descriptor of SAM's frozen embedding** (mean over the box tokens + mean over the crop, 768-d).
- **Memory** = descriptors + labels of all **training** candidates. **Memory score** = similarity-weighted share of real tumours among the 10 nearest memories. Training candidates are scored with a memory that **excludes their own patient** (5-fold by patient).
- Compared, same decision rule as S3d: **jury11** (S3d) vs **jury11 + memory** vs **jury11 + memory + probe**. Cut-off chosen on val, test once.

**Settings:** GPU T4 x2 · Internet ON. **Inputs:** cache · `cids-checkpoints`. **Time:** ~2 h (no HD95 → faster than S3d). Resumable per split.

In [ ]:
REPO_URL            = 'https://github.com/Vemuri-BK/cids.git'
GIT_BRANCH          = 'main'
EXPECTED_GIT_COMMIT = ''      # paste the full commit id you pushed

CACHE_ROOT = '/kaggle/input/datasets/kumarvemuri1/mama-mia-preprocessed-cache-2ch'
CKPT_DIR   = '/kaggle/input/datasets/kumarvemuri1/cids-checkpoints'
SEG_CKPT   = f'{CKPT_DIR}/segresnet_img_e100_best.pt'
SAM_CKPT   = f'{CKPT_DIR}/sam_s2b_best.pt'
CODE_DIR   = '/kaggle/working/cids'
OUT        = '/kaggle/working/s3e_memory'
K          = 10                # nearest memories
PATHS = [CACHE_ROOT, SEG_CKPT, SAM_CKPT]

In [ ]:
import os, subprocess
if os.path.isdir(f'{CODE_DIR}/.git'):
    subprocess.run(['git', '-C', CODE_DIR, 'fetch', 'origin', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'pull', 'origin', GIT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GIT_BRANCH, REPO_URL, CODE_DIR], check=True)
if EXPECTED_GIT_COMMIT:
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', EXPECTED_GIT_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', CODE_DIR, 'rev-parse', 'HEAD'], text=True).strip()
assert not EXPECTED_GIT_COMMIT or head == EXPECTED_GIT_COMMIT, (head, EXPECTED_GIT_COMMIT)
print('code at commit', head)
%cd {CODE_DIR}

def sh(cmd):
    """Run a shell step, stream its output, and STOP the notebook if it fails (plain !cmd does not).
    Progress bars redraw a line with '\r'; only each line's final state is printed (one line per epoch)."""
    import subprocess
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    buf = b''
    while True:
        chunk = p.stdout.read1(65536)
        if not chunk:
            break
        buf += chunk
        *lines, buf = buf.split(b'\n')
        for line in lines:
            print(line.split(b'\r')[-1].decode('utf-8', 'replace'), flush=True)
    if buf.strip():
        print(buf.split(b'\r')[-1].decode('utf-8', 'replace'), flush=True)
    if p.wait():
        raise RuntimeError(f'step failed (exit {p.returncode}): {cmd}')


In [ ]:
sh(f'pip install -q monai medim einops appdirs')
sh(f'python -m pytest -q tests')

In [ ]:
import torch
from pathlib import Path
assert torch.cuda.is_available(), 'No GPU! Settings -> Accelerator -> GPU T4 x2, then restart.'
for q in PATHS:
    print('OK ' if Path(q).exists() else '!! MISSING', q)
assert all(Path(q).exists() for q in PATHS)

## Smoke test (~6 min): 10 cases per split
Check: a `memory: N training candidates …` line, an AUC line, and a table with the three variants. Numbers on 10 cases mean nothing.

In [ ]:
sh(f'python scripts/s3e_memory_jury.py --cache_root {CACHE_ROOT} --seg_ckpt {SEG_CKPT} --sam_ckpt {SAM_CKPT} --out /kaggle/working/s3e_smoke --limit 10 --k 3')
sh(f'rm -rf /kaggle/working/s3e_smoke')

## Full run (~2 h)
**Read:** `jury11` should reproduce ≈ 0.735 on test. The memory helps if `jury11+memory` is higher on **val and test**, with `wilcoxon_p` < 0.05, and if `test_mem_knn` AUC is well above 0.5.

In [ ]:
sh(f'python scripts/s3e_memory_jury.py --cache_root {CACHE_ROOT} --seg_ckpt {SEG_CKPT} --sam_ckpt {SAM_CKPT} --out {OUT} --k {K}')

In [ ]:
import json
print(json.dumps(json.load(open(f'{OUT}/summary.json')), indent=2))
sh(f"""ls -la {OUT}""")